# Notebook 2 — Preprocessing Pipeline

Obiettivo: costruire e validare l'intera pipeline di preprocessing e salvare gli array processati in `data/processed/` per il riuso rapido nei notebook di training.

In [3]:
import numpy as np
import os
import sys
sys.path.append('..')
from src.preprocessing import (
    load_raw, add_rul, normalize_standard, normalize_clustered,
    make_windows, make_test_windows, get_feature_cols
)

DATA_DIR = '../data/raw'
OUT_DIR = '../data/processed'
os.makedirs(OUT_DIR, exist_ok=True)

# WINDOW_SIZE=30: ogni campione di input è una finestra di 30 cicli consecutivi.
# 30 cicli è un buon compromesso: abbastanza contesto per catturare il trend di degrado
# senza includere troppo rumore della fase iniziale sana.
WINDOW_SIZE = 30
# FD002 e FD004 hanno 6 condizioni operative → richiedono normalizzazione cluster-based
MULTI_COND = {'FD002', 'FD004'}

## Sliding Window

La **sliding window** (finestra scorrevole) è la tecnica centrale per trasformare
una serie temporale in campioni di training per una rete neurale:

```
Cicli:  [1, 2, 3, 4, 5, 6, ..., T]
           ←─ w=30 ─→             → campione 1: cicli 1-30,  label = RUL al ciclo 30
              ←─ w=30 ─→          → campione 2: cicli 2-31,  label = RUL al ciclo 31
                 ...
```

Ogni campione ha shape `(30, 17)` → 30 timestep, 17 feature (sensori + settings).
La label è il **RUL all'ultimo timestep** della finestra (non la media).

Vantaggi: data augmentation implicita (n_campioni >> n_motori), il modello vede
ogni punto della serie in molti contesti diversi.

## Normalizzazione: strategia diversa per dataset mono e multi-condizione

**FD001 e FD003** (1 condizione operativa): i sensori operano sempre nello stesso regime di volo → **MinMax globale**. Si scala ogni feature nell'intervallo [0, 1] fittando il normalizzatore solo sul train set per evitare data leakage.

**FD002 e FD004** (6 condizioni operative): richiedono la **normalizzazione cluster-based**.

**Il problema**: lo stesso sensore ha range di valori completamente diversi a seconda del regime di volo. Esempio: la temperatura di scarico vale 800–850°C a quota bassa e 600–650°C a quota alta. Con un MinMax globale, un valore normale a quota bassa e uno anomalo a quota alta ricevono la stessa scala — il modello non riesce a distinguere i due contesti operativi.

**La soluzione in tre passi**:
1. **KMeans(k=6)** sui 3 settings (quota, Mach, throttle) identifica i 6 regimi operativi e assegna ogni riga a un cluster
2. Per ogni cluster si calcola un **MinMax separato**, fittato solo sui dati di quel regime nel train
3. Ogni sensore viene scalato in [0, 1] **relativamente al suo regime** — un valore diventa alto o basso rispetto agli altri valori dello stesso contesto operativo, non rispetto a tutta la flotta

I centroidi appresi sul train vengono usati per assegnare anche il test set al cluster corretto → nessun data leakage.

In [4]:
for fd in ['FD001', 'FD002', 'FD003', 'FD004']:
    # 1. Carico i dati grezzi (26 colonne → drop sensori costanti → 19 colonne)
    train_df = load_raw(f'{DATA_DIR}/train_{fd}.txt')
    test_df  = load_raw(f'{DATA_DIR}/test_{fd}.txt')
    # RUL del test set: 1 valore per motore (quanti cicli mancavano alla fine del file di test)
    rul_test = np.loadtxt(f'{DATA_DIR}/RUL_{fd}.txt')

    # 2. Aggiungo la colonna RUL al train (cappata a 125)
    train_df = add_rul(train_df)
    feat_cols = get_feature_cols(train_df)  # esclude engine_id, cycle, rul

    # 3. Normalizzazione: strategia diversa in base al numero di condizioni operative
    if fd in MULTI_COND:
        # FD002/FD004: 6 regimi → KMeans(k=6) sui settings, poi MinMax per cluster
        train_df, test_df, _ = normalize_clustered(train_df, test_df, feat_cols)
    else:
        # FD001/FD003: 1 regime → MinMax globale (fit sul train, transform su entrambi)
        train_df, test_df, _ = normalize_standard(train_df, test_df, feat_cols)

    # 4. Sliding window: genera tutti i campioni (window_size, n_features) per il train
    X_train, y_train = make_windows(train_df, feat_cols, WINDOW_SIZE)
    # Per il test: solo l'ultima finestra per motore (1 predizione per motore)
    X_test = make_test_windows(test_df, feat_cols, WINDOW_SIZE)

    # 5. Salvo come .npy: caricamento rapido (np.load) senza ricalcolare tutta la pipeline
    np.save(f'{OUT_DIR}/X_train_{fd}.npy', X_train)
    np.save(f'{OUT_DIR}/y_train_{fd}.npy', y_train)
    np.save(f'{OUT_DIR}/X_test_{fd}.npy',  X_test)
    np.save(f'{OUT_DIR}/y_test_{fd}.npy',  rul_test.astype(np.float32))

    # Verifica shape: X_train deve essere (n_campioni, 30, 17)
    print(f'{fd}: X_train={X_train.shape}, y_train={y_train.shape}, X_test={X_test.shape}')

FD001: X_train=(17731, 30, 17), y_train=(17731,), X_test=(100, 30, 17)
FD002: X_train=(46219, 30, 17), y_train=(46219,), X_test=(259, 30, 17)
FD003: X_train=(21820, 30, 17), y_train=(21820,), X_test=(100, 30, 17)
FD004: X_train=(54028, 30, 17), y_train=(54028,), X_test=(248, 30, 17)


## Verifica dei risultati

Le shape attese sono:
- `X_train`: `(n_campioni, 30, 17)` — formato 3D richiesto da LSTM e Transformer
- `y_train`: `(n_campioni,)` — scalare per ogni finestra
- `X_test`: `(n_motori_test, 30, 17)` — 1 finestra per motore
- `y_test`: `(n_motori_test,)` — da `RUL_FDxxx.txt`

Il numero di campioni di train è molto maggiore del numero di motori grazie allo sliding window:
- FD001: 100 motori → ~17.731 campioni
- FD002: 260 motori → ~46.219 campioni
- FD003: 100 motori → ~21.820 campioni
- FD004: 249 motori → ~54.028 campioni